# Challenge 3: evaluación final e inferencia

El modelo ya fue seleccionado. Ahora debes abrir el conjunto de test una sola vez, comunicar su desempeño y comprobar que el pipeline puede reutilizarse con datos nuevos.

**Entradas:** `test.csv`, `data_contract.json` y `champion_model.joblib`  
**Salidas:** `test_metrics.json`, `batch_predictions.csv`

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 1. Carga del modelo y del conjunto de prueba

In [ ]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

## 2. Evaluación final

In [ ]:
# Evaluación final: usar el conjunto de test una sola vez.
y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy en test: {test_accuracy:.4f}")
print(f"F1 macro en test: {test_f1_macro:.4f}")
print("\\nReporte de clasificación:")
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Matriz de confusión — conjunto de test")
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(cm, index=[f"real_{c}" for c in model.classes_],
                     columns=[f"pred_{c}" for c in model.classes_]))


**Análisis:**

- El accuracy de test es **97,22 %** y el F1 macro es **97,10 %**.
- El accuracy promedio de validación cruzada fue **97,91 %**. El resultado de test es coherente y muy cercano al estimado en validación cruzada.
- La única confusión se da entre una observación de la clase real **2**, predicha como clase **1**. Las clases 0 y 1 se clasificaron correctamente en este conjunto.
- La pequeña diferencia entre validación cruzada y test no aporta evidencia clara de sobreajuste por sí sola. El conjunto de test tiene solo 36 registros, así que la estimación tiene incertidumbre.
- No se reajustan hiperparámetros después de observar test.


## 3. Función de inferencia

In [ ]:
feature_columns = contract["features"]

def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve predicción y probabilidades si están disponibles."""
    if not isinstance(records, pd.DataFrame):
        raise TypeError("records debe ser un pandas.DataFrame.")

    missing = [col for col in feature_columns if col not in records.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    # Ignora columnas adicionales y conserva el orden del entrenamiento.
    X_new = records.loc[:, feature_columns].copy()
    predictions = model.predict(X_new)
    result = pd.DataFrame({"prediction": predictions}, index=records.index)

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_new)
        for i, class_label in enumerate(model.classes_):
            result[f"probability_{class_label}"] = probabilities[:, i]
    return result


## 4. Predicción por lotes

In [ ]:
# Simulación de datos nuevos para demostrar la función de inferencia.
new_records = X_test.iloc[:5].copy()
batch_predictions = predict(new_records).reset_index(drop=True)
batch_path = REPORTS_DIR / "batch_predictions.csv"
batch_predictions.to_csv(batch_path, index=False)

display(batch_predictions)
print(f"Predicciones guardadas en: {batch_path}")


## 5. Reporte final

In [ ]:
# Guardar métricas finales del modelo campeón
metrics = {
    "model": training_metadata.get("champion_name", training_metadata.get("champion")),
    "selection_metric": training_metadata.get("selection_metric", "accuracy"),
    "accuracy_cv": float(training_metadata["best_cv_accuracy"]),
    "accuracy_test": float(test_accuracy),
    "f1_macro_test": float(test_f1_macro),
    "test_rows": int(len(y_test)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "classes": [int(c) if isinstance(c, (np.integer,)) else c for c in model.classes_],
    "note": "Test se utilizó para evaluación final, no para selección ni ajuste de hiperparámetros."
}
metrics_path = REPORTS_DIR / "test_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Métricas guardadas en: {metrics_path}")
print(json.dumps(metrics, indent=2, ensure_ascii=False))


In [ ]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

## Resumen ejecutivo

| Elemento | Resultado |
|---|---|
| Modelo campeón | Logistic Regression |
| Accuracy CV | 97,91 % |
| Accuracy test | 97,22 % |
| F1 macro test | 97,10 % |
| Clases más confundidas | Una muestra de la clase 2 fue predicha como clase 1 |
| Principal limitación | El test contiene 36 registros; el resultado puede variar con otras muestras |

## Checklist

- [x] Evalué únicamente el modelo campeón.
- [x] No reajusté hiperparámetros después de observar test.
- [x] Implementé validación básica del esquema de entrada.
- [x] Generé predicciones por lotes.
- [x] Guardé las métricas finales.
